In [1]:
from __future__ import annotations

import csv
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Iterator, Mapping, Protocol, runtime_checkable

In [2]:
# ---- Common types ----

Record = Mapping[str, Any]

In [3]:
# ---- Interface ----

@runtime_checkable
class BatchSource(Protocol):
    """Reads data from files and yields records or batches."""
    def read(self) -> Iterator[Record]: ...
    # optionally:
    # def read_batches(self) -> Iterator[list[Record]]: ...


In [4]:
# ---- CSV plugin ----

@dataclass(frozen=True)
class CsvBatchSource(BatchSource):
    path: str | Path
    delimiter: str = ","
    encoding: str = "utf-8"
    newline: str = ""

    def read(self) -> Iterator[Record]:
        """
        Yields one record (row) at a time as a dict: {column_name: value}.
        Values are strings by default (that's how csv works).
        """
        path = Path(self.path)
        with path.open(mode="r", encoding=self.encoding, newline=self.newline) as f:
            reader = csv.DictReader(f, delimiter=self.delimiter)
            for row in reader:
                # row is a dict[str, str | None]; we return it as Mapping[str, Any]
                yield row

    def read_batches(self, batch_size: int = 1000) -> Iterator[list[Record]]:
        """
        Optional: yields lists of records (batches).
        Useful if downstream validators/checkers work in chunks.
        """
        if batch_size <= 0:
            raise ValueError("batch_size must be > 0")

        batch: list[Record] = []
        for rec in self.read():
            batch.append(rec)
            if len(batch) >= batch_size:
                yield batch
                batch = []
        if batch:
            yield batch


In [5]:
# ---- Example usage ----
def engine(source: BatchSource) -> int:
    processed = 0
    for record in source.read():
        # do something with record
        processed += 1
    return processed

In [6]:

# Example:
src = CsvBatchSource("data/users.csv", delimiter=";")
engine(src)

20000

In [7]:
for batch in src.read_batches(batch_size=5):
    print("batch size:", len(batch))
    print(batch)

batch size: 5
[{'id,email,age': '1.0,christy82@example.com,999'}, {'id,email,age': '2.0,francisco39@example.org,83'}, {'id,email,age': '3.0,nicholsrichard@example.org,28'}, {'id,email,age': '4.0,aallen@example.net,36'}, {'id,email,age': '5.0,katherinestokes@example.org,28'}]
batch size: 5
[{'id,email,age': '6.0,hmorris@example.net,35'}, {'id,email,age': '7.0,paulfernandez@example.org,90'}, {'id,email,age': '8.0,not-an-email,5'}, {'id,email,age': '9.0,meganshelton@example.com,61'}, {'id,email,age': '10.0,herringkatrina@example.com,72'}]
batch size: 5
[{'id,email,age': '11.0,nmorales@example.org,81'}, {'id,email,age': '12.0,bwebb@example.org,80'}, {'id,email,age': '13.0,matthew65@example.org,2'}, {'id,email,age': '14.0,singhwilliam@example.net,1'}, {'id,email,age': '15.0,hector59@example.org,5'}]
batch size: 5
[{'id,email,age': '16.0,imitchell@example.org,45'}, {'id,email,age': '17.0,george13@example.net,10'}, {'id,email,age': '18.0,glennnoah@example.org,45'}, {'id,email,age': '19.0,lmoo

In [8]:
import json

@dataclass(frozen=True)
class JsonlBatchSource(BatchSource):
    """
    Reads JSON Lines (JSONL): one JSON object per line.

    Compatible with input like:
    {"id": 1, "first_name": "...", ...}
    {"id": 2, "first_name": "...", ...}
    """
    path: str | Path
    encoding: str = "utf-8"

    def read(self) -> Iterator[Record]:
        path = Path(self.path)
        with path.open("r", encoding=self.encoding) as f:
            for line_no, line in enumerate(f, start=1):
                line = line.strip()
                if not line:
                    continue  # skip empty lines
                try:
                    obj = json.loads(line)
                except json.JSONDecodeError as e:
                    raise ValueError(f"Invalid JSON on line {line_no} in {path}: {e}") from e

                if not isinstance(obj, dict):
                    raise ValueError(
                        f"Expected a JSON object (dict) on line {line_no} in {path}, "
                        f"got {type(obj).__name__}"
                    )

                yield obj

    def read_batches(self, batch_size: int = 1000) -> Iterator[list[Record]]:
        if batch_size <= 0:
            raise ValueError("batch_size must be > 0")

        batch: list[Record] = []
        for rec in self.read():
            batch.append(rec)
            if len(batch) >= batch_size:
                yield batch
                batch = []
        if batch:
            yield batch

In [9]:
src = JsonlBatchSource("data/users.jsonl")
engine(src)

2000

In [10]:
for b in src.read_batches(5):
    print("batch:", len(b), "first_id:", b[0]["id"])
    print(b)

batch: 5 first_id: 1
[{'id': 1, 'first_name': 'Danielle', 'last_name': 'Johnson', 'email': 'john21@example.net', 'created_at': '2011-07-01T17:26:53.752269', 'is_active': True, 'country': 'BT', 'system_x_id': '619176'}, {'id': 2, 'first_name': 'Robert', 'last_name': 'Johnson', 'email': 'jesseguzman@example.net', 'created_at': '1981-02-20T22:24:36.834725', 'is_active': True, 'country': 'ME', 'system_x_id': '571412'}, {'id': 3, 'first_name': 'Robert', 'last_name': 'Cole', 'email': 'lisa02@example.net', 'created_at': '1993-09-05T05:04:05.210034', 'is_active': True, 'country': 'DE', 'system_x_id': '225772'}, {'id': 4, 'first_name': 'Susan', 'last_name': 'Rogers', 'email': 'jamesmichael@example.com', 'created_at': '2003-10-29T09:45:50.507730', 'is_active': True, 'country': 'ZA', 'system_x_id': '481741'}, {'id': 5, 'first_name': 'Amanda', 'last_name': 'Dudley', 'email': 'smiller@example.net', 'created_at': '2004-08-27T03:21:00.977319', 'is_active': True, 'country': 'TR', 'system_x_id': '20162

**What is happening here?**

```
src = CsvBatchSource("data/users.csv", delimiter=";")
engine(src)
```

```
src = JsonlBatchSource("data/users.jsonl")
engine(src)
```


In both cases, src is an object that **knows how to read data**, but **does not read anything yet**.

-   CsvBatchSource(...)  knows how to read rows from a CSV file

-   JsonlBatchSource(...)  knows how to read records from a JSON Lines file




Both classes implement the same  **interface**:

```
class BatchSource(Protocol):
    def read(self) -> Iterator[Record]: ...
```

This means:



> _“If an object has a_ _read()_ _method that yields records, it can be used as a BatchSource.”_
